In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         PIPELINE DE ENGENHARIA DE DADOS                      ║
# ║          Transformação, Consolidação e Auditoria dos Dados                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Data        : Fevereiro / 2026
#  Versão      : 3.0
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de transformação e consolidação da arquitetura
#  de dados do TCC. Ele realiza o processamento, padronização e exportação dos
#  19 conjuntos de dados brutos coletados na etapa anterior (script de coleta),
#  organizando-os por submercado elétrico (NORTE, NORDESTE, SUL, SUDESTE) e
#  estruturando-os em séries horárias contínuas para uso em modelos preditivos.
#
#  O pipeline aplica as seguintes transformações por tipo de dado:
#    • Horários    : padronização de colunas, saneamento e agregação por subsistema
#    • Diários     : expansão via cross-join × 24h (forward fill horário)
#    • Semanais    : explosão de intervalos em dias → horas (média)
#    • Semi-horários: filtro minuto=00 e resampling para frequência horária
#    • Meteorológicos: agregação espacial de estações → média regional por sub.
#
#  Ao final da execução, o script gera:
#    (a) arquivos CSV por submercado e dataset em DIR_SAIDA;
#    (b) Tabela 2 — Volumetria consolidada (registros horários por submercado);
#    (c) Tabela 3 — Estratégias de transformação por dataset;
#    (d) Tabela 4 — Dimensão final e complexidade por submercado.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script consome os arquivos CSV gerados pelo script de coleta (Versão 3.0).
#  Os parâmetros de controle estão definidos nas constantes abaixo:
#
#    ANOS_ALVO   : lista de anos a serem processados  (padrão: 2021–2026)
#    BASE_PATH   : caminho raiz de armazenamento
#                  → Google Drive (/content/drive/MyDrive/TCC_PLD_Project)
#                  → local (./TCC_DATA_MASTER) se Drive não estiver montado
#    DIR_ENTRADA : BASE_PATH / "2-DADOS" / "1-DADOS-BRUTOS"
#    DIR_SAIDA   : BASE_PATH / "2-DADOS" / "2-DADOS-AGRUPADOS"
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Arquivos CSV  →  DIR_SAIDA / <pasta_dataset> / <NN>_<FONTE>_<SUB>_<NOME>.csv
#     Convenção de nomenclatura: <num_ds>_<fonte>_<submercado>_<dataset>.csv
#     Exemplo: 01_CCEE_SUDESTE_PLD.csv
#     Separador: vírgula (,) | Encoding: UTF-8 | Sem índice de linha
#
#  2. Saída no terminal (ASCII colorido):
#     • Tabela 2 — Volumetria consolidada (registros horários × submercado)
#     • Tabela 3 — Estratégias de transformação e consolidação por dataset
#     • Tabela 4 — Dimensão final e complexidade por submercado
#     • Log de avisos acumulados durante o processamento
#     • Tempo total de execução
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DATASETS PROCESSADOS (resumo)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  N°  Chave               Origem  Estratégia             Descrição
#  01  PLD                 CCEE    Padronização + seleção  Preço de Liquidação das Diferenças
#  02  INTERCAMBIO         ONS     Saldo líquido orig/dest Intercâmbio Nacional entre subsistemas
#  03  BALANCO             ONS     Saneamento + agregação  Balanço Energético por subsistema
#  04  CMO                 ONS     Filtro min=00 + resamp  Custo Marginal de Operação
#  05  CURVA               ONS     Saneamento + agregação  Curva de Carga do sistema
#  06  EAR                 ONS     Cross-join × 24h        Energia Armazenada nos reservatórios
#  07  ENA                 ONS     Cross-join × 24h        Energia Natural Afluente
#  08  CARGA               ONS     Cross-join × 24h        Carga de Energia por subsistema
#  09  CVU                 ONS     Explode semana → horas  Custo Variável Unitário (termelétricas)
#  10  VOLUME_ESPERA       ONS     Cross-join × 24h        Volume de Espera nos reservatórios
#  11  HIDRO               ONS     Saneamento + agregação  Dados Hidrológicos dos reservatórios
#  12  DISPONIBILIDADE     ONS     Saneamento + agregação  Disponibilidade das usinas
#  13  FATOR_CAPACIDADE    ONS     Cast numérico + groupby Fator de Capacidade das usinas
#  14  GERACAO             ONS     Saneamento + agregação  Geração por usina
#  15  VERTIDA             ONS     Saneamento + agregação  Energia Vertida e Turbinável
#  16  TERMICA             ONS     Auto-detect val_* cols  Geração Térmica de despacho
#  17  INMET               INMET   Agregação espacial      Variáveis meteorológicas (estações)
#  18  CARGA_VERIFICADA    ONS     Filtro min=00 + mapa    Carga verificada via API REST
#  19  DOLAR               BCB     Broadcast diário × 24h Cotação PTAX do dólar americano
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_PATH.
#     Antes de executar, montar o Google Drive:
#
#         from google.colab import drive
#         drive.mount('/content/drive')
#
#  2. PRÉ-REQUISITO
#     Este script depende dos arquivos gerados pelo script de coleta (v3.0).
#     Certifique-se de que a etapa de coleta foi concluída com sucesso antes
#     de executar este pipeline de transformação.
#
#  3. CORREÇÕES APLICADAS (v3.0)
#     • processar_pld(): Ramo A detecta MES_REFERENCIA + DIA + HORA ANTES de
#       qualquer tentativa de parsing datetime. Parser próprio _extrair_ano_mes()
#       lida com formatos AAAAMM, AAMM e variantes.
#     • processar_pld(): coluna 'referencia' excluída da busca de col_dt para
#       não capturar MES_REFERENCIA como datetime (causava KEY_ANO nulo).
#     • processar_pld(): PERIODO_COMERCIALIZACAO excluído da busca de coluna de
#       submercado (falso positivo nas versões anteriores).
#     • processar_pld(): diagnóstico rico imprime KEY_ANO range, submercados
#       únicos e estatísticas de PLD_HORA antes de salvar.
#
#  4. PADRONIZAÇÃO DE COLUNAS-CHAVE
#     Todos os arquivos de saída compartilham o mesmo conjunto de colunas-chave:
#       KEY_SUBMERCADO, KEY_ANO, KEY_MES, KEY_DIA, KEY_HORA
#     As colunas de feature recebem prefixo padronizado:
#       <num_ds>_<fonte>_<submercado>_<TIPO>_<NOME_VAR>
#
#  5. LEITURA FLEXÍVEL DE CSV
#     A função ler_csv_seguro() tenta automaticamente separadores (;  e ,) e
#     encodings (UTF-8 e latin-1), com diferentes casas decimais (vírgula e
#     ponto). Linhas mal formadas são ignoradas (on_bad_lines="skip").
#
#  6. MAPEAMENTO DE SUBMERCADOS
#     Siglas regionais da ONS são normalizadas para nomes completos via
#     SIGLA_TO_SUB: {"N":"NORTE", "NE":"NORDESTE", "S":"SUL",
#     "SE":"SUDESTE", "CO":"SUDESTE", "SECO":"SUDESTE", ...}
#
#  7. EXPANSÃO DE FREQUÊNCIA (DIÁRIO → HORÁRIO)
#     Datasets de frequência diária (EAR, ENA, CARGA, VOLUME_ESPERA) são
#     expandidos via cross-join com um DataFrame de 24 horas. O valor diário
#     é replicado para todas as horas do dia (forward fill implícito).
#
#  8. GERENCIAMENTO DE MEMÓRIA
#     DataFrames intermediários são explicitamente deletados e gc.collect()
#     é chamado após cada processador. O INMET processa estações em chunks
#     de 30 para evitar MemoryError em ambientes com RAM limitada (Colab).
#
#  9. DADOS DO ANO CORRENTE (2026)
#     Arquivos do ano em curso podem estar incompletos. O campo de volumetria
#     na Tabela 2 refletirá apenas os dados disponíveis até a data de execução.
#
#  10. REPRODUTIBILIDADE
#      Os nomes dos arquivos de saída seguem convenção determinística baseada
#      em número do dataset, fonte, submercado e nome do dataset, garantindo
#      rastreabilidade total e sobrescrita segura em reexecuções.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas auxiliares
#  pandas            1.5            Manipulação de DataFrames e séries temporais
#  gc                —              Gerenciamento explícito de memória
#  unicodedata       —              Normalização de nomes de colunas (INMET)
#  re                —              Expressões regulares para detecção de colunas
#  calendar          —              Cálculo de horas esperadas por ano
#
#  (numpy, pandas e demais são stdlib ou já disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TCC_PLD_Project/
#  └── 2-DADOS/
#      └── 2-DADOS-AGRUPADOS/
#          ├── 01-PLD/          01_CCEE_{SUB}_PLD.csv           (×4 submercados)
#          ├── 02-INTERCAMBIO/  02_ONS_{SUB}_INTERCAMBIO.csv
#          ├── 03-BALANCO-ENERGIA/
#          ├── 04-CMO/
#          ├── 05-CURVA/
#          ├── 06-EAR/
#          ├── 07-ENA/
#          ├── 08-CARGA/
#          ├── 09-CVU/
#          ├── 10-VOLUME-ESPERA/
#          ├── 11-HIDRO/
#          ├── 12-DISPONIBILIDADE/
#          ├── 13-FATOR-CAPACIDADE/
#          ├── 14-GERACAO/
#          ├── 15-VERTIDA/
#          ├── 16-TERMICA/
#          ├── 17-METEOROLOGIA/  17_INMET_{SUB}_METEOROLOGIA.csv
#          ├── 18-CARGA-VERIFICADA/
#          └── 19-DOLAR/
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Executar o script:
#      exec(open('arquitetura_dados_tcc_v10_2.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import os
import numpy as np
import warnings
import gc
import unicodedata
import re
import time
import calendar
from datetime import datetime as dt

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================
BASE_PATH = r"/content/drive/MyDrive/TCC_PLD_Project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "./TCC_DATA_MASTER"

DIR_ENTRADA = os.path.join(BASE_PATH, "2-DADOS/1-DADOS-BRUTOS")
DIR_SAIDA   = os.path.join(BASE_PATH, "2-DADOS/2-DADOS-AGRUPADOS")

ANOS_ALVO = [2021, 2022, 2023, 2024, 2025, 2026]
HORAS_ESPERADAS = sum(366 * 24 if calendar.isleap(y) else 365 * 24 for y in ANOS_ALVO)

# Dicionário de estatísticas por submercado — preenchido durante o processamento
STATS = {
    "NORTE":    {"linhas": 0, "cols_oper": 0, "cols_clima": 0, "cols_fin": 0, "cols_target": 0,
                 "estacoes": 0, "arquivos": 0, "bytes": 0, "tempo": 0.0},
    "NORDESTE": {"linhas": 0, "cols_oper": 0, "cols_clima": 0, "cols_fin": 0, "cols_target": 0,
                 "estacoes": 0, "arquivos": 0, "bytes": 0, "tempo": 0.0},
    "SUL":      {"linhas": 0, "cols_oper": 0, "cols_clima": 0, "cols_fin": 0, "cols_target": 0,
                 "estacoes": 0, "arquivos": 0, "bytes": 0, "tempo": 0.0},
    "SUDESTE":  {"linhas": 0, "cols_oper": 0, "cols_clima": 0, "cols_fin": 0, "cols_target": 0,
                 "estacoes": 0, "arquivos": 0, "bytes": 0, "tempo": 0.0},
}

# ==============================================================================
# 📁 MAPEAMENTO DE PASTAS (entrada → saída)
# ==============================================================================
MAPA_PASTAS = {
    "PLD":              (os.path.join(DIR_ENTRADA, "1-CCEE-PLD-HORARIO"),
                         os.path.join(DIR_SAIDA,   "01-PLD")),
    "INTERCAMBIO":      (os.path.join(DIR_ENTRADA, "7-ONS-INTERCAMBIO-ENERGIA-NACIONAL"),
                         os.path.join(DIR_SAIDA,   "02-INTERCAMBIO")),
    "BALANCO":          (os.path.join(DIR_ENTRADA, "2-ONS-BALANÇO-DE-ENERGIA-HORARIO"),
                         os.path.join(DIR_SAIDA,   "03-BALANCO-ENERGIA")),
    "CMO":              (os.path.join(DIR_ENTRADA, "5-ONS-CMO-HORARIO"),
                         os.path.join(DIR_SAIDA,   "04-CMO")),
    "CURVA":            (os.path.join(DIR_ENTRADA, "8-ONS-CURVA-CARGA"),
                         os.path.join(DIR_SAIDA,   "05-CURVA")),
    "EAR":              (os.path.join(DIR_ENTRADA, "12-ONS-EAR"),
                         os.path.join(DIR_SAIDA,   "06-EAR")),
    "ENA":              (os.path.join(DIR_ENTRADA, "13-ONS-ENA"),
                         os.path.join(DIR_SAIDA,   "07-ENA")),
    "CARGA":            (os.path.join(DIR_ENTRADA, "14-ONS-CARGA"),
                         os.path.join(DIR_SAIDA,   "08-CARGA")),
    "CVU":              (os.path.join(DIR_ENTRADA, "15-ONS-CVU"),
                         os.path.join(DIR_SAIDA,   "09-CVU")),
    "VOLUME_ESPERA":    (os.path.join(DIR_ENTRADA, "17-ONS-VOLUME-ESPERA"),
                         os.path.join(DIR_SAIDA,   "10-VOLUME-ESPERA")),
    "HIDRO":            (os.path.join(DIR_ENTRADA, "3-ONS-HIDRAULICOS-RESERVATORIOS-HORARIO"),
                         os.path.join(DIR_SAIDA,   "11-HIDRO")),
    "DISPONIBILIDADE":  (os.path.join(DIR_ENTRADA, "9-ONS-DISPONIBILIDADE-USINA"),
                         os.path.join(DIR_SAIDA,   "12-DISPONIBILIDADE")),
    "FATOR_CAPACIDADE": (os.path.join(DIR_ENTRADA, "18-ONS-FATOR-CAPACIDADE"),
                         os.path.join(DIR_SAIDA,   "13-FATOR-CAPACIDADE")),
    "GERACAO":          (os.path.join(DIR_ENTRADA, "4-ONS-GERACAO-USINA-HORARIO"),
                         os.path.join(DIR_SAIDA,   "14-GERACAO")),
    "VERTIDA":          (os.path.join(DIR_ENTRADA, "6-ONS-ENERGIA-VERTIDA-TURBINAVEL"),
                         os.path.join(DIR_SAIDA,   "15-VERTIDA")),
    "TERMICA":          (os.path.join(DIR_ENTRADA, "10-ONS-GERACAO-TERMICA-DESPACHO"),
                         os.path.join(DIR_SAIDA,   "16-TERMICA")),
    "INMET":            (os.path.join(DIR_ENTRADA, "11-INMET-METEOROLOGICOS"),
                         os.path.join(DIR_SAIDA,   "17-METEOROLOGIA")),
    "CARGA_VERIFICADA": (os.path.join(DIR_ENTRADA, "16-ONS-CARGA-VERIFICADA"),
                         os.path.join(DIR_SAIDA,   "18-CARGA-VERIFICADA")),
    "DOLAR":            (os.path.join(DIR_ENTRADA, "19-BCB-DOLAR"),
                         os.path.join(DIR_SAIDA,   "19-DOLAR")),
}

SUBMERCADOS = ["NORTE", "NORDESTE", "SUL", "SUDESTE"]

# Mapeamento de siglas regionais da ONS para nomes completos de submercado
SIGLA_TO_SUB = {
    "N": "NORTE", "NE": "NORDESTE", "S": "SUL", "SE": "SUDESTE",
    "CO": "SUDESTE", "SECO": "SUDESTE", "SUDESTE/CENTRO-OESTE": "SUDESTE",
    "SE/CO": "SUDESTE",
}

# ==============================================================================
# 🏷️ METADADOS DE DATASET
# ==============================================================================
DATASET_NUM = {
    "PLD":              "01",
    "INTERCAMBIO":      "02",
    "BALANCO":          "03",
    "CMO":              "04",
    "CURVA":            "05",
    "EAR":              "06",
    "ENA":              "07",
    "CARGA":            "08",
    "CVU":              "09",
    "VOLUME_ESPERA":    "10",
    "HIDRO":            "11",
    "DISPONIBILIDADE":  "12",
    "FATOR_CAPACIDADE": "13",
    "GERACAO":          "14",
    "VERTIDA":          "15",
    "TERMICA":          "16",
    "INMET":            "17",
    "CARGA_VERIFICADA": "18",
    "DOLAR":            "19",
}

DATASET_FONTE = {
    "PLD":              "CCEE",
    "INTERCAMBIO":      "ONS",
    "BALANCO":          "ONS",
    "CMO":              "ONS",
    "CURVA":            "ONS",
    "EAR":              "ONS",
    "ENA":              "ONS",
    "CARGA":            "ONS",
    "CVU":              "ONS",
    "VOLUME_ESPERA":    "ONS",
    "HIDRO":            "ONS",
    "DISPONIBILIDADE":  "ONS",
    "FATOR_CAPACIDADE": "ONS",
    "GERACAO":          "ONS",
    "VERTIDA":          "ONS",
    "TERMICA":          "ONS",
    "INMET":            "INMET",
    "CARGA_VERIFICADA": "ONS",
    "DOLAR":            "BCB",
}

# ==============================================================================
# 📋 METADADOS DAS TABELAS DE RELATÓRIO
# ==============================================================================
TABELA2_META = [
    ("01", "PLD",              "CCEE",  "Preço (PLD) CCEE"),
    ("02", "INTERCAMBIO",      "ONS",   "Intercâmbio Nacional"),
    ("03", "BALANCO",          "ONS",   "Balanço Energético"),
    ("04", "CMO",              "ONS",   "Custo Marginal (CMO)"),
    ("05", "CURVA",            "ONS",   "Curva de Carga"),
    ("06", "EAR",              "ONS",   "Energia Armazenada (EAR)"),
    ("07", "ENA",              "ONS",   "Energia Natural (ENA)"),
    ("08", "CARGA",            "ONS",   "Carga de Energia"),
    ("09", "CVU",              "ONS",   "Custo Variável (CVU)"),
    ("10", "VOLUME_ESPERA",    "ONS",   "Volume de Espera"),
    ("11", "HIDRO",            "ONS",   "Dados Hidrológicos"),
    ("12", "DISPONIBILIDADE",  "ONS",   "Disponibilidade Usina"),
    ("13", "FATOR_CAPACIDADE", "ONS",   "Fator Capacidade"),
    ("14", "GERACAO",          "ONS",   "Geração Usinas"),
    ("15", "VERTIDA",          "ONS",   "Energia Vertida"),
    ("16", "TERMICA",          "ONS",   "Geração Térmica"),
    ("17", "INMET",            "INMET", "Meteorologia (INMET)"),
    ("18", "CARGA_VERIFICADA", "ONS",   "Carga Verificada (API)"),
    ("19", "DOLAR",            "BCB",   "Cotação Dólar"),
]

TABELA3_META = [
    ("01", "Preço (PLD) CCEE",        "CCEE – TARGET",             "Horária",       "Padronização submercado + seleção col.",  "Série-alvo horária contínua"),
    ("02", "Intercâmbio Nacional",    "ONS – Feature Operacional", "Horária",       "Cálculo de saldo líquido (origem/dest.)", "Fluxo líquido horário por sub."),
    ("03", "Balanço Energético",      "ONS – Feature Operacional", "Horária",       "Saneamento, cast numérico & agregação",   "Geração+carga horária por sub."),
    ("04", "Custo Marginal (CMO)",    "ONS – Feature Operacional", "Semi-Horária",  "Filtro minuto=00 + agregação horária",    "CMO horário síncrono por sub."),
    ("05", "Curva de Carga",          "ONS – Feature Operacional", "Horária",       "Saneamento & agregação por subsistema",   "Curva de carga horária por sub."),
    ("06", "Energia Armazenada (EAR)","ONS – Feature Operacional", "Diária",        "Cross-join × 24h (Forward Fill diário)",  "EAR expandida hora a hora"),
    ("07", "Energia Natural (ENA)",   "ONS – Feature Operacional", "Diária",        "Cross-join × 24h (Forward Fill diário)",  "ENA expandida hora a hora"),
    ("08", "Carga de Energia",        "ONS – Feature Operacional", "Diária",        "Cross-join × 24h (Forward Fill diário)",  "Carga expandida hora a hora"),
    ("09", "Custo Variável (CVU)",    "ONS – Feature Operacional", "Semanal",       "Explode semana → dias → 24h (média)",     "CVU médio diário horário"),
    ("10", "Volume de Espera",        "ONS – Feature Operacional", "Diária",        "Cross-join × 24h (Forward Fill diário)",  "Volume expandido hora a hora"),
    ("11", "Dados Hidrológicos",      "ONS – Feature Operacional", "Horária",       "Saneamento & agregação por reservatório", "Vazão/volume horário por sub."),
    ("12", "Disponibilidade Usina",   "ONS – Feature Operacional", "Mensal",        "Saneamento & agregação mensal→horária",   "Potência instalada mensal/hora"),
    ("13", "Fator Capacidade",        "ONS – Feature Operacional", "Horária",       "Cast numérico, groupby & soma horária",   "Fatores de capacidade horários"),
    ("14", "Geração Usinas",          "ONS – Feature Operacional", "Horária",       "Saneamento & agregação por usina/sub.",   "Geração total horária por sub."),
    ("15", "Energia Vertida",         "ONS – Feature Operacional", "Horária",       "Saneamento & agregação por subsistema",   "Energia vertida horária por sub."),
    ("16", "Geração Térmica",         "ONS – Feature Operacional", "Horária",       "Auto-detect cols val_* + map-reduce",     "Geração térmica horária por sub."),
    ("17", "Meteorologia (INMET)",    "INMET – Feature Meteorol.", "Horária (raw)", "Agregação espacial estações → região",    "Média regional horária por sub."),
    ("18", "Carga Verificada (API)",  "ONS – Feature Operacional", "Horária",       "Filtro minuto=00, mapeamento sigla/sub.", "Carga verificada horária por sub."),
    ("19", "Cotação Dólar",           "BCB – Feature Financeira",  "Diária",        "Broadcast diário × 24h × 4 submercados", "Cotação PTAX horária por sub."),
]

# ==============================================================================
# 🎨 CLASSE DE INTERFACE & RELATÓRIOS (REPORT MANAGER)
# ==============================================================================
class UI:
    RESET   = "\033[0m";  BOLD    = "\033[1m";  CYAN    = "\033[96m"
    GREEN   = "\033[92m"; YELLOW  = "\033[93m"; RED     = "\033[91m"
    BLUE    = "\033[94m"; MAGENTA = "\033[95m"

    WARNINGS_BUFFER = []
    VOLUMETRY_DATA  = {}   # {chave_ds: {sub: linhas}}

    @staticmethod
    def banner():
        print(f"{UI.BLUE}")
        print("╔════════════════════════════════════════════════════════════════════╗")
        print("║   🏛️  PIPELINE DE ENGENHARIA DE DADOS - SUNTEB & TCC MASTER V10.2  ║")
        print("║   Fix processar_pld(): KEY_ANO vazio causado por MES_REFERENCIA   ║")
        print("╚════════════════════════════════════════════════════════════════════╝")
        print(f"{UI.RESET}")

    @staticmethod
    def step(msg):
        print(f"{UI.CYAN}➤ {msg}{UI.RESET}")

    @staticmethod
    def success(msg, detalhe=""):
        print(f"   {UI.GREEN}✔ {msg:<55} {UI.RESET}{UI.BOLD}{detalhe}{UI.RESET}")

    @staticmethod
    def warning(msg):
        print(f"   {UI.YELLOW}⚠ {msg}{UI.RESET}")

    @staticmethod
    def error(msg):
        print(f"   {UI.RED}✖ ERRO CRÍTICO: {msg}{UI.RESET}")

    @staticmethod
    def register_warning(dataset, regiao):
        UI.WARNINGS_BUFFER.append(f"[{dataset}] Sem dados para região: {regiao}")

    @staticmethod
    def log_volumetry(chave_ds, submercado, linhas):
        if chave_ds not in UI.VOLUMETRY_DATA:
            UI.VOLUMETRY_DATA[chave_ds] = {s: 0 for s in SUBMERCADOS}
        UI.VOLUMETRY_DATA[chave_ds][submercado] = max(
            UI.VOLUMETRY_DATA[chave_ds].get(submercado, 0), linhas)

    # ── Tabela 2 — Volumetria Consolidada ──────────────────────────────────────
    @staticmethod
    def print_tabela_2():
        W = 148
        subs = SUBMERCADOS
        print(f"\n{UI.BOLD}Tabela 2 – Volumetria Consolidada dos Dados Operativos (Registros Horários){UI.RESET}")
        print(f"Período Alvo: {min(ANOS_ALVO)} a {max(ANOS_ALVO)} | "
              f"Horas Esperadas por Submercado: {HORAS_ESPERADAS:,}")
        print(f"{UI.MAGENTA}╔{'═'*W}╗")
        header = (f"║ {'Nº':^3} │ {'Fonte':^5} │ {'VARIÁVEL (DATASET)':<27} "
                  f"║ {'NORTE':^9} {'FALT':^8} "
                  f"║ {'NORDESTE':^9} {'FALT':^8} "
                  f"║ {'SUL':^9} {'FALT':^8} "
                  f"║ {'SUDESTE':^9} {'FALT':^8} "
                  f"║ {'COBERTURA':^10} ║")
        print(header)
        sep = (f"╠{'═'*3}╪{'═'*7}╪{'═'*29}"
               f"╬{'═'*10}{'═'*9}╬{'═'*10}{'═'*9}╬{'═'*10}{'═'*9}╬{'═'*10}{'═'*9}╬{'═'*12}╣")
        print(sep)

        for num, chave_ds, fonte, rotulo in TABELA2_META:
            vols = UI.VOLUMETRY_DATA.get(chave_ds, {s: 0 for s in subs})
            cols_sub = []
            total_regs = 0
            subs_ativos = 0
            for s in subs:
                regs = vols.get(s, 0)
                falt = max(0, HORAS_ESPERADAS - regs) if regs > 0 else HORAS_ESPERADAS
                cols_sub.append((regs, falt))
                if regs > 0:
                    total_regs  += regs
                    subs_ativos += 1
            cob = ((total_regs / subs_ativos) / HORAS_ESPERADAS) * 100 if subs_ativos > 0 else 0.0
            if cob >= 99.0:
                cob_str = f"{UI.GREEN}{cob:6.1f}%{UI.RESET}"
            elif cob < 80.0:
                cob_str = f"{UI.RED}{cob:6.1f}%{UI.RESET}"
            else:
                cob_str = f"{cob:6.1f}%"
            linha = f"║ {num:^3} │ {fonte:^5} │ {rotulo:<27} "
            for regs, falt in cols_sub:
                r_str = f"{regs:>9,}" if regs > 0 else f"{'–':^9}"
                f_str = f"{falt:>8,}" if falt > 0 else f"{'–':^8}"
                linha += f"║ {r_str} {f_str} "
            linha += f"║ {cob_str:^20} ║"
            print(linha)
        print(f"╚{'═'*W}╝{UI.RESET}")

    # ── Tabela 3 — Estratégias de Transformação ────────────────────────────────
    @staticmethod
    def print_tabela_3():
        C = {"num": 4, "rotulo": 26, "tipologia": 30, "freq": 16, "estrategia": 44, "resultado": 36}
        W = sum(C.values()) + 13
        print(f"\n{UI.BOLD}Tabela 3 – Estratégias de Transformação e Consolidação por Dataset{UI.RESET}")
        print(f"{UI.MAGENTA}╔{'═'*W}╗")
        print(f"║ {'Nº':^{C['num']}} │ {'VARIÁVEL':<{C['rotulo']}} │ {'TIPOLOGIA':<{C['tipologia']}} "
              f"│ {'FREQ':<{C['freq']}} │ {'ESTRATÉGIA':<{C['estrategia']}} │ {'RESULTADO':<{C['resultado']}} ║")
        print(f"╠{'═'*(C['num']+2)}╪{'═'*(C['rotulo']+2)}╪{'═'*(C['tipologia']+2)}"
              f"╪{'═'*(C['freq']+2)}╪{'═'*(C['estrategia']+2)}╪{'═'*(C['resultado']+2)}╣")
        for num, rotulo, tipologia, freq, estrategia, resultado in TABELA3_META:
            print(f"║ {num:^{C['num']}} │ {rotulo:<{C['rotulo']}} │ {tipologia:<{C['tipologia']}} "
                  f"│ {freq:<{C['freq']}} │ {estrategia:<{C['estrategia']}} │ {resultado:<{C['resultado']}} ║")
        print(f"╚{'═'*W}╝{UI.RESET}")

    # ── Tabela 4 — Dimensão Final por Submercado ───────────────────────────────
    @staticmethod
    def print_tabela_4():
        C = {"sub": 12, "est": 13, "target": 8, "oper": 11, "meteo": 11, "fin": 9, "total": 9}
        W = sum(C.values()) + 14
        print(f"\n{UI.BOLD}Tabela 4 – Dimensão Final e Complexidade por Submercado{UI.RESET}")
        print(f"{UI.MAGENTA}╔{'═'*W}╗")
        print(f"║ {'SUBMERCADO':<{C['sub']}} ║ {'Nº ESTAÇÕES':^{C['est']}} ║ {'TARGET':^{C['target']}} "
              f"║ {'FEAT OPER':^{C['oper']}} ║ {'FEAT METEO':^{C['meteo']}} ║ {'FEAT FIN':^{C['fin']}} "
              f"║ {'TOTAL':^{C['total']}} ║")
        print(f"╠{'═'*(C['sub']+2)}╬{'═'*(C['est']+2)}╬{'═'*(C['target']+2)}╬{'═'*(C['oper']+2)}"
              f"╬{'═'*(C['meteo']+2)}╬{'═'*(C['fin']+2)}╬{'═'*(C['total']+2)}╣")
        for reg, s in STATS.items():
            total = s['cols_target'] + s['cols_oper'] + s['cols_clima'] + s['cols_fin']
            print(f"║ {reg:<{C['sub']}} ║ {s['estacoes']:^{C['est']}} ║ {s['cols_target']:^{C['target']}} "
                  f"║ {s['cols_oper']:^{C['oper']}} ║ {s['cols_clima']:^{C['meteo']}} "
                  f"║ {s['cols_fin']:^{C['fin']}} ║ {total:^{C['total']}} ║")
        print(f"╚{'═'*W}╝{UI.RESET}")


# ==============================================================================
# 🔧 UTILITÁRIOS DE LEITURA & PADRONIZAÇÃO
# ==============================================================================

def ler_csv_seguro(caminho):
    """
    Lê CSV com múltiplos fallbacks de separador, encoding e decimal.
    Tenta as combinações: ; utf-8 vírgula → , utf-8 ponto → ; latin-1 vírgula → , latin-1 ponto.
    Retorna DataFrame vazio se todas as tentativas falharem.
    """
    for sep, enc, dec in [(';', 'utf-8', ','), (',', 'utf-8', '.'),
                           (';', 'latin-1', ','), (',', 'latin-1', '.')]:
        try:
            df = pd.read_csv(caminho, sep=sep, encoding=enc,
                             on_bad_lines='skip', decimal=dec, low_memory=False)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return pd.DataFrame()


def criar_tempo(df, col_data, col_hora=None):
    """
    Extrai as chaves temporais KEY_ANO, KEY_MES, KEY_DIA, KEY_HORA
    a partir de uma coluna de data/datetime.
    Se col_hora for informada, usa seus valores diretos para KEY_HORA;
    caso contrário, extrai a hora da própria coluna de data.
    Retorna o DataFrame com as colunas-chave adicionadas.
    """
    if df.empty or col_data not in df.columns:
        candidatos = [x for x in df.columns
                      if any(k in x.lower() for k in ('data', 'instante', 'referencia'))]
        if candidatos:
            col_data = candidatos[0]
        else:
            return df
    df = df.copy()
    df[col_data] = pd.to_datetime(df[col_data], errors='coerce').dt.tz_localize(None)
    df['KEY_ANO']  = df[col_data].dt.year.fillna(0).astype("int32")
    df['KEY_MES']  = df[col_data].dt.month.fillna(0).astype("int32")
    df['KEY_DIA']  = df[col_data].dt.day.fillna(0).astype("int32")
    if col_hora and col_hora in df.columns:
        df['KEY_HORA'] = pd.to_numeric(df[col_hora], errors='coerce').fillna(0).astype("int32")
    else:
        df['KEY_HORA'] = df[col_data].dt.hour.fillna(0).astype("int32")
    return df


# Conjunto das colunas-chave padronizadas — usado para separar keys de features
_KEY_COLS = frozenset(
    ['KEY_SUBMERCADO', 'KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA'])


def _limpar_nome_variavel(col):
    """Remove prefixos TARGET_ e FEATURE_ do nome da coluna e converte para maiúsculas."""
    nome = re.sub(r'^(TARGET_|FEATURE_)', '', col, flags=re.IGNORECASE)
    return nome.upper()


def _renomear_features(df, num_ds, fonte, sub, tipo_dado):
    """
    Renomeia colunas de feature para o padrão:
      <num_ds>_<fonte>_<sub>_<TIPO>_<NOME_VAR>
    As colunas-chave (_KEY_COLS) não são renomeadas.
    """
    mapa = {}
    for col in df.columns:
        if col in _KEY_COLS:
            continue
        nome_var = _limpar_nome_variavel(col)
        mapa[col] = f"{num_ds}_{fonte}_{sub}_{tipo_dado}_{nome_var}"
    return df.rename(columns=mapa)


def listar_csvs(pasta, prefixo):
    """
    Lista arquivos CSV em 'pasta' cujo nome começa com 'prefixo' (case-insensitive).
    Retorna lista ordenada de caminhos completos, ou lista vazia se a pasta
    não existir ou não houver arquivos correspondentes.
    """
    if not os.path.exists(pasta):
        return []
    return sorted([
        os.path.join(pasta, f)
        for f in os.listdir(pasta)
        if f.lower().endswith(".csv") and f.lower().startswith(prefixo.lower())
    ])


def salvar_por_submercado(df, pasta_saida, nome_base,
                           col_sub="KEY_SUBMERCADO",
                           forcar_sub=None,
                           chave_ds="",
                           num_ds="00",
                           fonte="ONS",
                           tipo_dado="FEATURE"):
    """
    Particiona df por submercado e salva um CSV por região em pasta_saida.
    Aplica normalização de siglas via SIGLA_TO_SUB, renomeia features com
    _renomear_features() e atualiza os contadores de STATS e VOLUMETRY_DATA.

    Parâmetros:
      forcar_sub  : se informado, processa apenas esse submercado (uso no INMET).
      chave_ds    : chave do dataset para atualizar UI.VOLUMETRY_DATA.
      tipo_dado   : "TARGET", "FEATURE", "CLIMA" ou "FIN" — controla qual
                    contador de STATS é incrementado.
    """
    if df is None or df.empty:
        return

    os.makedirs(pasta_saida, exist_ok=True)

    if col_sub in df.columns:
        df = df.copy()
        df[col_sub] = (df[col_sub].astype(str)
                                   .str.upper()
                                   .str.strip()
                                   .replace(SIGLA_TO_SUB))

    lista_subs = [forcar_sub] if forcar_sub else SUBMERCADOS

    for sub in lista_subs:
        t0 = time.time()

        if col_sub in df.columns:
            df_sub = df[df[col_sub] == sub].copy()
        else:
            df_sub = df.copy()

        if df_sub.empty:
            UI.register_warning(nome_base, sub)
            continue

        cols_keys  = [c for c in _KEY_COLS if c in df_sub.columns]
        cols_feats = [c for c in df_sub.columns if c not in _KEY_COLS]
        df_sub     = df_sub[cols_keys + cols_feats]

        tipo_col = "TARGET" if tipo_dado == "TARGET" else "FEATURE"
        df_sub = _renomear_features(df_sub, num_ds, fonte, sub, tipo_col)

        nome_arquivo = f"{num_ds}_{fonte}_{sub}_{nome_base.upper()}.csv"
        caminho      = os.path.join(pasta_saida, nome_arquivo)
        df_sub.to_csv(caminho, index=False)

        n_feats = len(cols_feats)
        STATS[sub]["arquivos"] += 1
        STATS[sub]["bytes"]    += os.path.getsize(caminho)
        STATS[sub]["tempo"]    += (time.time() - t0)
        STATS[sub]["linhas"]    = max(STATS[sub]["linhas"], len(df_sub))

        if tipo_dado == "TARGET":
            STATS[sub]["cols_target"] += n_feats
        elif tipo_dado == "CLIMA":
            STATS[sub]["cols_clima"]   = n_feats
        elif tipo_dado == "FIN":
            STATS[sub]["cols_fin"]     = max(STATS[sub]["cols_fin"], n_feats)
        else:
            STATS[sub]["cols_oper"]    = max(STATS[sub]["cols_oper"],
                                            STATS[sub]["cols_oper"] + max(0, n_feats - 5))

        if chave_ds:
            UI.log_volumetry(chave_ds, sub, len(df_sub))

        UI.success(f"Salvo: {nome_arquivo}", f"({len(df_sub):,} linhas | {n_feats} feats)")

    del df
    gc.collect()


# ==============================================================================
# 🧩 ETL GENÉRICO
# ==============================================================================

def etl_simples(chave_mapa, prefixo_arq, col_feature_map, nome_base,
                filtro_hora00=False, chave_ds="", freq="Horária", tipo_dado="FEATURE"):
    """
    Processador genérico para datasets horários com estrutura padrão ONS.
    Aplica o fluxo: ler → criar chaves temporais → cast numérico →
    groupby por subsistema/hora → salvar por submercado.

    Parâmetros:
      col_feature_map : dict {nome_col_original: nome_col_saida}
      filtro_hora00   : se True, mantém apenas registros com minuto == 0 (ex.: CMO)
      freq            : frequência do dado (apenas para log)
    """
    num_ds = DATASET_NUM[chave_mapa]
    fonte  = DATASET_FONTE[chave_mapa]
    UI.step(f"[{num_ds}] Processando {chave_mapa} (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave_mapa]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, prefixo_arq)
    if not arquivos:
        UI.warning(f"Sem arquivos com prefixo '{prefixo_arq}' em: {dir_in}")
        UI.warning(f"  Arquivos existentes: {os.listdir(dir_in) if os.path.exists(dir_in) else '—'}")
        return

    dfs_agregados = []
    for arquivo in arquivos:
        df_temp = ler_csv_seguro(arquivo)
        if df_temp.empty:
            continue

        col_dt = next((c for c in df_temp.columns
                       if any(k in c.lower() for k in ('instante', 'data', 'referencia'))), None)
        if col_dt:
            df_temp = criar_tempo(df_temp, col_dt)
            if filtro_hora00:
                df_temp[col_dt] = pd.to_datetime(df_temp[col_dt], errors='coerce')
                df_temp = df_temp[df_temp[col_dt].dt.minute == 0]

        cols_uteis = [c for c in col_feature_map if c in df_temp.columns]
        if not cols_uteis:
            UI.warning(f"  Colunas {list(col_feature_map.keys())} não achadas em "
                       f"{os.path.basename(arquivo)}")
            UI.warning(f"  Disponíveis: {list(df_temp.columns[:10])}")
            continue

        for c in cols_uteis:
            if df_temp[c].dtype == object:
                df_temp[c] = pd.to_numeric(
                    df_temp[c].astype(str).str.replace(',', '.'), errors='coerce')
            df_temp[c] = pd.to_numeric(df_temp[c], errors='coerce').astype('float32')

        col_sub_orig = next((c for c in df_temp.columns
                             if 'subsistema' in c.lower() or 'submercado' in c.lower()), None)
        if not col_sub_orig:
            UI.warning(f"Coluna de subsistema ausente em {os.path.basename(arquivo)}");  continue

        grp_cols = [c for c in [col_sub_orig, "KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"]
                    if c in df_temp.columns]
        df_agg = df_temp.groupby(grp_cols, as_index=False)[cols_uteis].sum()
        df_agg.rename(columns={col_sub_orig: 'KEY_SUBMERCADO'}, inplace=True)
        dfs_agregados.append(df_agg)
        del df_temp, df_agg;  gc.collect()

    if not dfs_agregados:
        UI.warning(f"Nenhum dado válido encontrado para {chave_mapa}");  return

    df_final = pd.concat(dfs_agregados, ignore_index=True)
    feat_cols = [c for c in col_feature_map if c in df_final.columns]
    df_final  = df_final.groupby(
        ["KEY_SUBMERCADO", "KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"],
        as_index=False)[feat_cols].sum()

    df_final.rename(columns=col_feature_map, inplace=True)

    salvar_por_submercado(
        df_final, dir_out, nome_base,
        chave_ds=chave_ds, num_ds=num_ds, fonte=fonte, tipo_dado=tipo_dado)

    del df_final, dfs_agregados;  gc.collect()


# ==============================================================================
# 🧩 PROCESSADORES ESPECÍFICOS
# ==============================================================================

# ── 01 – PLD ──────────────────────────────────────────────────────────────────
def processar_pld():
    """
    Processador do PLD (Preço de Liquidação das Diferenças — CCEE).

    PROBLEMA CORRIGIDO (v3.0):
      Versões anteriores capturavam MES_REFERENCIA como coluna datetime porque
      'referencia' estava na lista de keywords. pd.to_datetime() retornava NaT,
      KEY_ANO ficava nulo e dropna() esvaziava todo o DataFrame.

    Esquemas de data aceitos (auto-detectados em ordem de prioridade):
      A) MES_REFERENCIA (AAAAMM) + DIA + HORA  ← formato padrão dos arquivos CCEE
      B) Coluna datetime completa (instante / datahora / timestamp)
      C) MES_REFERENCIA isolado (sem DIA/HORA explícitas)
      D) Qualquer coluna de data como último fallback

    Colunas de valor aceitas (regex ampliado):
      valor | pld | preco | price | mwh | r$ | brl | custo | tarifa

    Coluna de submercado:
      Exclui PERIODO_COMERCIALIZACAO (falso positivo identificado na v2.x).
    """
    chave  = "PLD"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando PLD (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    # ── Verifica existência da pasta de entrada ────────────────────────────────
    if not os.path.exists(dir_in):
        UI.error(f"Pasta PLD não encontrada: {dir_in}")
        return

    # ── Lista todos os arquivos CSV com prefixo "pld" ─────────────────────────
    arquivos = listar_csvs(dir_in, "pld")
    if not arquivos:
        UI.error(f"Nenhum arquivo CSV começando com 'pld' encontrado em: {dir_in}")
        UI.warning(f"  Conteúdo da pasta: {os.listdir(dir_in)}")
        return

    UI.step(f"   ↳ {len(arquivos)} arquivo(s) PLD encontrado(s):")
    for a in arquivos:
        UI.step(f"      • {os.path.basename(a)}")

    # ── Leitura e concatenação de todos os arquivos ───────────────────────────
    dfs = []
    for caminho_arq in arquivos:
        df_temp = ler_csv_seguro(caminho_arq)
        if df_temp.empty:
            UI.warning(f"  Arquivo vazio ou ilegível: {os.path.basename(caminho_arq)}")
            continue
        UI.success(f"  Lido: {os.path.basename(caminho_arq)}",
                   f"({len(df_temp):,} linhas × {df_temp.shape[1]} colunas)")
        dfs.append(df_temp)

    if not dfs:
        UI.error("Todos os arquivos PLD estão vazios ou ilegíveis.")
        return

    df = pd.concat(dfs, ignore_index=True)
    df.columns = [str(c).strip() for c in df.columns]
    UI.step(f"   ↳ Total combinado: {len(df):,} linhas | colunas: {list(df.columns)}")

    # ══════════════════════════════════════════════════════════════════════════
    # CAMADA 1 – Montagem das chaves temporais
    #
    # Ordem de prioridade:
    #   A) MES_REFERENCIA (AAAAMM) + DIA + HORA  ← formato real dos arquivos CCEE
    #   B) Coluna datetime completa (instante / datahora / timestamp)
    #      *** 'referencia' EXCLUÍDO desta busca para não capturar MES_REFERENCIA ***
    #   C) MES_REFERENCIA isolado (sem DIA/HORA explícitas)
    #   D) Qualquer coluna genérica de data como último recurso
    # ══════════════════════════════════════════════════════════════════════════

    # Mapa upper → nome original para busca case-insensitive
    df_upper = {c.upper(): c for c in df.columns}

    # Localiza MES_REFERENCIA e variantes no cabeçalho
    col_mes_ref_orig = df_upper.get('MES_REFERENCIA')
    if col_mes_ref_orig is None:
        col_mes_ref_orig = next(
            (df_upper[cu] for cu in df_upper if 'MES' in cu and 'REF' in cu), None)

    col_dia_orig  = df_upper.get('DIA')
    col_hora_orig = df_upper.get('HORA')

    # ── Ramo A: MES_REFERENCIA + DIA + HORA (formato CCEE padrão) ─────────────
    if col_mes_ref_orig is not None:
        amostra_mes = df[col_mes_ref_orig].dropna().astype(str).head(3).tolist()
        UI.step(f"   ↳ Ramo A – MES_REFERENCIA: '{col_mes_ref_orig}' | amostra: {amostra_mes}")

        mes_str = (df[col_mes_ref_orig]
                   .astype(str)
                   .str.strip()
                   .str.replace('.', '', regex=False)
                   .str.replace('-', '', regex=False)
                   .str.replace('/', '', regex=False))

        def _extrair_ano_mes(s):
            """
            Parser robusto para o campo MES_REFERENCIA da CCEE.
            Aceita AAAAMM (6 dígitos) e AAMM (4 dígitos legacy).
            Retorna tupla (ano, mes) ou (None, None) se o formato for inválido.
            """
            s = str(s).strip()
            if len(s) >= 6 and s[:6].isdigit():
                return int(s[:4]), int(s[4:6])
            elif len(s) == 4 and s.isdigit():
                return 2000 + int(s[:2]), int(s[2:4])
            return None, None

        pares = mes_str.apply(_extrair_ano_mes)
        df['KEY_ANO'] = pd.array([p[0] for p in pares], dtype="Int32")
        df['KEY_MES'] = pd.array([p[1] for p in pares], dtype="Int32")

        if col_dia_orig and col_dia_orig in df.columns:
            df['KEY_DIA'] = pd.to_numeric(df[col_dia_orig], errors='coerce').astype("Int32")
        else:
            df['KEY_DIA'] = pd.array([1] * len(df), dtype="Int32")

        if col_hora_orig and col_hora_orig in df.columns:
            df['KEY_HORA'] = pd.to_numeric(df[col_hora_orig], errors='coerce').astype("Int32")
        else:
            df['KEY_HORA'] = pd.array([0] * len(df), dtype="Int32")

        validos = df['KEY_ANO'].notna().sum()
        UI.step(f"   ↳ KEY_ANO range: {df['KEY_ANO'].min()}–{df['KEY_ANO'].max()} "
                f"| linhas com ANO válido: {validos:,}/{len(df):,}")

    else:
        # ── Ramo B: coluna datetime completa ──────────────────────────────────
        # IMPORTANTE: exclui colunas com 'mes' no nome para não capturar MES_REFERENCIA
        col_dt = next(
            (c for c in df.columns
             if any(k in c.lower()
                    for k in ('instante', 'datahora', 'datetime', 'timestamp'))
             and 'mes' not in c.lower()),
            None
        )
        if col_dt is None:
            col_dt = next(
                (c for c in df.columns
                 if 'data' in c.lower() and 'mes' not in c.lower()),
                None
            )

        if col_dt:
            UI.step(f"   ↳ Ramo B – datetime completo: '{col_dt}'")
            df = criar_tempo(df, col_dt)

        else:
            # ── Ramo C: MES_REFERENCIA sem DIA/HORA ───────────────────────────
            col_mes_any = next(
                (c for c in df.columns if 'mes' in c.lower() and 'refer' in c.lower()), None)
            if col_mes_any:
                UI.step(f"   ↳ Ramo C – MES_REFERENCIA isolado: '{col_mes_any}'")
                s2 = (df[col_mes_any].astype(str).str.strip()
                      .str.replace('.', '', regex=False))
                df['KEY_ANO']  = pd.to_numeric(s2.str[:4], errors='coerce').astype("Int32")
                df['KEY_MES']  = pd.to_numeric(s2.str[4:6], errors='coerce').astype("Int32")
                df['KEY_DIA']  = pd.array([1] * len(df), dtype="Int32")
                df['KEY_HORA'] = pd.array([0] * len(df), dtype="Int32")
            else:
                # ── Ramo D: última tentativa ───────────────────────────────────
                col_any = next(
                    (c for c in df.columns
                     if any(k in c.lower() for k in ('data', 'date', 'ano', 'year'))), None)
                if col_any:
                    UI.step(f"   ↳ Ramo D – fallback genérico: '{col_any}'")
                    df = criar_tempo(df, col_any)
                else:
                    UI.error(f"Impossível identificar coluna temporal no PLD. "
                             f"Colunas disponíveis: {list(df.columns)}")
                    return

    # ══════════════════════════════════════════════════════════════════════════
    # CAMADA 2 – Detecção da coluna de submercado
    # Exclui PERIODO_COMERCIALIZACAO (falso positivo identificado na v2.x)
    # ══════════════════════════════════════════════════════════════════════════
    col_sub = next(
        (c for c in df.columns
         if any(k in c.lower()
                for k in ('subsistema', 'submercado', 'nom_sub',
                          'sub', 'mercado', 'regiao', 'sistema', 'area'))
         and 'KEY'     not in c.upper()
         and 'PERIODO' not in c.upper()
         and 'COMERCI' not in c.upper()),
        None
    )

    if col_sub is None:
        UI.error(f"Coluna de submercado não encontrada no PLD. "
                 f"Colunas disponíveis: {list(df.columns)}")
        return

    UI.step(f"   ↳ Coluna de submercado: '{col_sub}'")
    df.rename(columns={col_sub: 'KEY_SUBMERCADO'}, inplace=True)
    df['KEY_SUBMERCADO'] = (df['KEY_SUBMERCADO']
                            .astype(str).str.upper().str.strip()
                            .replace(SIGLA_TO_SUB))

    # Renomeia DIA/HORA originais caso ainda não tenham sido renomeadas (Ramo A)
    for alias, key in [('DIA', 'KEY_DIA'), ('HORA', 'KEY_HORA'),
                       ('dia', 'KEY_DIA'), ('hora', 'KEY_HORA')]:
        if alias in df.columns and key not in df.columns:
            df.rename(columns={alias: key}, inplace=True)

    # ══════════════════════════════════════════════════════════════════════════
    # CAMADA 3 – Detecção da coluna de valor PLD
    # Regex ampliado para cobrir nomenclaturas variadas dos arquivos CCEE
    # ══════════════════════════════════════════════════════════════════════════
    col_val = next(
        (c for c in df.columns
         if re.search(r'(valor|pld|preco|price|mwh|r\$|brl|custo|tarifa)',
                      c, re.IGNORECASE)
         and 'KEY' not in c.upper()),
        None
    )

    if col_val is None:
        # Fallback: primeira coluna numérica com >50% de valores válidos
        for c in df.columns:
            if c in _KEY_COLS or c == 'KEY_SUBMERCADO':
                continue
            try:
                serie = pd.to_numeric(
                    df[c].astype(str).str.replace(',', '.'), errors='coerce')
                if serie.notna().sum() > len(df) * 0.5:
                    col_val = c
                    UI.warning(f"  Coluna de valor PLD inferida automaticamente: '{c}'")
                    break
            except Exception:
                continue

    if col_val is None:
        UI.error(f"Coluna de valor PLD não encontrada. "
                 f"Colunas disponíveis: {list(df.columns)}")
        return

    UI.step(f"   ↳ Coluna de valor PLD identificada: '{col_val}'")

    df[col_val] = pd.to_numeric(
        df[col_val].astype(str).str.replace(',', '.'), errors='coerce')

    nome_feat = "PLD_HORA"
    df.rename(columns={col_val: nome_feat}, inplace=True)

    # ══════════════════════════════════════════════════════════════════════════
    # MONTAGEM E LIMPEZA FINAL
    # ══════════════════════════════════════════════════════════════════════════
    cols_obrig = ['KEY_SUBMERCADO', 'KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA', nome_feat]
    cols_exist = [c for c in cols_obrig if c in df.columns]

    colunas_faltantes = [c for c in cols_obrig if c not in df.columns]
    if colunas_faltantes:
        UI.warning(f"  Colunas ausentes no PLD final: {colunas_faltantes}")

    df_final = df[cols_exist].copy()

    # Remove linhas sem ANO/MES ou sem valor PLD válido
    subset_drop = [c for c in ['KEY_ANO', 'KEY_MES', nome_feat] if c in df_final.columns]
    df_final = df_final.dropna(subset=subset_drop)

    # Converte chaves para int após dropna (seguro pois NaN já foi removido)
    for kc in ['KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA']:
        if kc in df_final.columns:
            df_final[kc] = pd.to_numeric(df_final[kc], errors='coerce').fillna(0).astype(int)

    # Filtra apenas os anos definidos em ANOS_ALVO
    if 'KEY_ANO' in df_final.columns:
        df_final = df_final[df_final['KEY_ANO'].isin(ANOS_ALVO)]

    # ── Diagnóstico pré-salvamento ─────────────────────────────────────────────
    UI.step(f"   ↳ Registros válidos para salvar: {len(df_final):,}")
    if not df_final.empty:
        UI.step(f"   ↳ KEY_ANO únicos  : {sorted(df_final['KEY_ANO'].unique().tolist())}")
        UI.step(f"   ↳ KEY_SUBMERCADO  : {sorted(df_final['KEY_SUBMERCADO'].unique().tolist())}")
        UI.step(f"   ↳ PLD_HORA  min={df_final[nome_feat].min():.2f}  "
                f"max={df_final[nome_feat].max():.2f}  "
                f"mean={df_final[nome_feat].mean():.2f}")

    if df_final.empty:
        UI.error("DataFrame PLD ficou vazio após limpeza. Verifique os dados fonte.")
        UI.warning("  → Amostra do df antes do dropna (para debug):")
        try:
            UI.warning(str(df[cols_exist].head(5).to_string()))
        except Exception:
            pass
        return

    salvar_por_submercado(
        df_final, dir_out, "pld",
        chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="TARGET")

    del df, df_final, dfs
    gc.collect()


# ── 02 – Intercâmbio Nacional ─────────────────────────────────────────────────
def processar_intercambio():
    """
    Processa o Intercâmbio Nacional entre subsistemas (ONS).
    Calcula o saldo líquido horário por subsistema: subtrai os fluxos de saída
    (origem) e soma os fluxos de entrada (destino) para cada região.
    Fallback: agrega diretamente pela coluna de subsistema se origem/destino
    não estiverem disponíveis.
    """
    chave  = "INTERCAMBIO"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando Intercâmbio (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "intercambio_ons")
    if not arquivos:
        arquivos = listar_csvs(dir_in, "intercambio")
    if not arquivos:
        UI.warning(f"Sem arquivos de intercâmbio em: {dir_in}");  return

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return

    df = pd.concat(dfs, ignore_index=True)
    col_dt = next((c for c in df.columns
                   if any(k in c.lower() for k in ('instante', 'data', 'referencia'))), None)
    if col_dt:
        df = criar_tempo(df, col_dt)

    val_col  = next((c for c in df.columns if 'intercambio' in c.lower() and 'val' in c.lower()),
                    'val_intercambiomwmed')
    col_orig = next((c for c in df.columns if 'origem' in c.lower()), None)
    col_dest = next((c for c in df.columns if 'destino' in c.lower()), None)

    if col_orig and col_dest and val_col in df.columns:
        # Saldo líquido: saída = negativo (origem), entrada = positivo (destino)
        origem          = df.copy()
        origem["_sub"]  = df[col_orig].astype(str).str.upper()
        origem["_val"]  = -pd.to_numeric(df[val_col], errors='coerce')
        destino         = df.copy()
        destino["_sub"] = df[col_dest].astype(str).str.upper()
        destino["_val"] = pd.to_numeric(df[val_col], errors='coerce')

        df_f = (pd.concat([
                    origem[["_sub","KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA","_val"]],
                    destino[["_sub","KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA","_val"]]])
                  .groupby(["_sub","KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA"], as_index=False)
                  ["_val"].sum())
        df_f.rename(columns={'_sub': 'KEY_SUBMERCADO',
                             '_val': 'VAL_INTERCAMBIOMED'}, inplace=True)
    else:
        col_sub = next((c for c in df.columns
                        if 'subsistema' in c.lower() or 'submercado' in c.lower()), None)
        if not col_sub or val_col not in df.columns:
            UI.error(f"Estrutura inesperada. Colunas: {list(df.columns)}");  return
        df_f = df.groupby([col_sub,"KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA"],
                          as_index=False)[val_col].sum()
        df_f.rename(columns={col_sub: 'KEY_SUBMERCADO',
                             val_col: 'VAL_INTERCAMBIOMED'}, inplace=True)

    df_f['KEY_SUBMERCADO'] = (df_f['KEY_SUBMERCADO'].astype(str)
                                                     .str.upper()
                                                     .replace(SIGLA_TO_SUB))

    salvar_por_submercado(df_f, dir_out, "intercambio",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df, df_f;  gc.collect()


# ── 06/07/08/10 – Expansão diária → horária ───────────────────────────────────
def expandir_diario(chave, prefixo_arq, col_val_candidatos, _unused_prefix="FEATURE"):
    """
    Expande datasets de frequência diária para frequência horária via cross-join.
    O valor diário é replicado para as 24 horas do dia (equivalente a forward fill).
    Aceita lista de nomes candidatos para a coluna de valor (tenta em ordem).
    Usado pelos datasets: EAR (06), ENA (07), CARGA (08), VOLUME_ESPERA (10).
    """
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando {chave} – expansão diária (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, prefixo_arq)
    if not arquivos:
        UI.warning(f"Sem arquivos com prefixo '{prefixo_arq}' em: {dir_in}");  return

    if isinstance(col_val_candidatos, str):
        col_val_candidatos = [col_val_candidatos]

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        UI.warning(f"Todos os arquivos de {chave} estão vazios");  return

    df = pd.concat(dfs, ignore_index=True)

    col_data = next((c for c in df.columns
                     if any(k in c.lower() for k in ('data', 'instante', 'referencia'))), None)
    if not col_data:
        UI.warning(f"Coluna de data não encontrada para {chave}. Colunas: {list(df.columns)}");  return

    df = criar_tempo(df, col_data)
    # Remove KEY_HORA gerada por criar_tempo — será recriada via cross-join
    if 'KEY_HORA' in df.columns:
        df = df.drop(columns=['KEY_HORA'])

    col_val = next((c for c in col_val_candidatos if c in df.columns), None)
    if not col_val:
        col_val = next((c for c in df.columns
                        if any(cv.split('_')[-1] in c.lower() for cv in col_val_candidatos)
                        and c not in ('KEY_ANO', 'KEY_MES', 'KEY_DIA')), None)
    if not col_val:
        UI.warning(f"Coluna de valor não encontrada para {chave}. "
                   f"Candidatos: {col_val_candidatos}. Disponíveis: {list(df.columns)}");  return

    col_sub = next((c for c in df.columns
                    if 'subsistema' in c.lower() or 'submercado' in c.lower()), None)
    if not col_sub:
        UI.warning(f"Coluna de subsistema não encontrada para {chave}.");  return

    df[col_val] = pd.to_numeric(
        df[col_val].astype(str).str.replace(',', '.'), errors='coerce').astype('float32')

    # Cross-join com as 24 horas do dia
    h        = pd.DataFrame({'KEY_HORA': range(24)})
    df['_k'] = 1;  h['_k'] = 1
    df = pd.merge(df, h, on='_k').drop('_k', axis=1)

    nome_feat_intermediario = col_val.upper()
    df_out = df.rename(columns={col_sub: 'KEY_SUBMERCADO',
                                  col_val: nome_feat_intermediario})
    df_out['KEY_SUBMERCADO'] = (df_out['KEY_SUBMERCADO'].astype(str)
                                                         .str.upper()
                                                         .replace(SIGLA_TO_SUB))

    salvar_por_submercado(
        df_out, dir_out,
        chave.lower().replace("_", "-"),
        chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")

    del df, df_out;  gc.collect()


# ── 09 – CVU ──────────────────────────────────────────────────────────────────
def processar_cvu():
    """
    Processa o Custo Variável Unitário das termelétricas (ONS).
    Expande intervalos semanais (dat_iniciosemana → dat_fimsemana) em dias
    e depois em horas via cross-join, calculando a média do CVU por hora.
    """
    chave  = "CVU"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando CVU (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "cvu_")
    if not arquivos:
        arquivos = listar_csvs(dir_in, "cvu")
    if not arquivos:
        UI.warning(f"Sem arquivos CVU em: {dir_in}");  return

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return

    df = pd.concat(dfs, ignore_index=True)

    col_ini = next((c for c in df.columns if 'inicio' in c.lower() and 'semana' in c.lower()),
                   'dat_iniciosemana')
    col_fim = next((c for c in df.columns if 'fim' in c.lower() and 'semana' in c.lower()),
                   'dat_fimsemana')
    col_cvu = next((c for c in df.columns if 'cvu' in c.lower()), 'val_cvu')
    col_sub = next((c for c in df.columns if 'subsistema' in c.lower()
                    or 'submercado' in c.lower()), 'nom_subsistema')

    df[col_ini] = pd.to_datetime(df[col_ini], errors='coerce')
    df[col_fim] = pd.to_datetime(df[col_fim], errors='coerce')
    if col_cvu in df.columns:
        df[col_cvu] = pd.to_numeric(
            df[col_cvu].astype(str).str.replace(',', '.'), errors='coerce')

    # Explosão de semana → lista de dias → linhas individuais
    df = df.dropna(subset=[col_ini, col_fim])
    df['list_dates'] = [pd.date_range(s, e, freq='D').tolist()
                        for s, e in zip(df[col_ini], df[col_fim])]
    df = df.explode('list_dates')
    df['KEY_ANO'] = df['list_dates'].dt.year
    df['KEY_MES'] = df['list_dates'].dt.month
    df['KEY_DIA'] = df['list_dates'].dt.day

    # Cross-join com as 24 horas do dia
    h = pd.DataFrame({'KEY_HORA': range(24)})
    df['_k'] = 1;  h['_k'] = 1
    df = pd.merge(df, h, on='_k').drop('_k', axis=1)

    grp_cols = [c for c in [col_sub,'KEY_ANO','KEY_MES','KEY_DIA','KEY_HORA']
                if c in df.columns]
    df_agg = df.groupby(grp_cols)[col_cvu].mean().reset_index()
    nome_feat = col_cvu.upper()
    df_agg.rename(columns={col_sub: 'KEY_SUBMERCADO', col_cvu: nome_feat}, inplace=True)
    df_agg['KEY_SUBMERCADO'] = (df_agg['KEY_SUBMERCADO'].astype(str)
                                                          .str.upper()
                                                          .replace(SIGLA_TO_SUB))

    salvar_por_submercado(df_agg, dir_out, "cvu",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df, df_agg;  gc.collect()


# ── 13 – Fator de Capacidade ──────────────────────────────────────────────────
def processar_fator_capacidade():
    """
    Processa o Fator de Capacidade das usinas (ONS).
    Detecta automaticamente as colunas de valor (val_*) e realiza
    cast numérico + groupby horário + soma por subsistema.
    """
    chave  = "FATOR_CAPACIDADE"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando Fator de Capacidade (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "fator_capacidade")
    if not arquivos:
        UI.warning(f"Sem arquivos fator_capacidade*.csv em: {dir_in}");  return

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return

    df = pd.concat(dfs, ignore_index=True)
    col_dt = next((c for c in df.columns
                   if any(k in c.lower() for k in ('instante','data','referencia'))), None)
    if col_dt:
        df = criar_tempo(df, col_dt)

    col_sub = next((c for c in df.columns
                    if 'subsistema' in c.lower() or 'submercado' in c.lower()), 'nom_subsistema')
    df['KEY_SUBMERCADO'] = (df[col_sub].astype(str).str.upper().replace(SIGLA_TO_SUB))

    # Colunas de valor conhecidas; fallback para qualquer col val_*
    cols_alvo = ['val_geracaoprogramada','val_geracaoverificada',
                 'val_capacidadeinstalada','val_fatorcapacidade']
    cols_exist = [c for c in cols_alvo if c in df.columns]
    if not cols_exist:
        cols_exist = [c for c in df.columns if c.startswith('val_')]

    for c in cols_exist:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.'), errors='coerce')

    grp = [c for c in ['KEY_SUBMERCADO','KEY_ANO','KEY_MES','KEY_DIA','KEY_HORA']
           if c in df.columns]
    df_ag = df.groupby(grp)[cols_exist].sum().reset_index()
    df_ag.rename(columns={c: c.upper() for c in cols_exist}, inplace=True)

    salvar_por_submercado(df_ag, dir_out, "fator-capacidade",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df, df_ag;  gc.collect()


# ── 16 – Geração Térmica ──────────────────────────────────────────────────────
def processar_termica():
    """
    Processa a Geração Térmica de despacho (ONS).
    Detecta automaticamente todas as colunas val_* disponíveis em cada arquivo
    (estrutura pode variar entre anos) e realiza map-reduce por subsistema/hora.
    """
    chave  = "TERMICA"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando Geração Térmica (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "geracao_termica")
    if not arquivos:
        UI.warning(f"Sem arquivos geracao_termica*.csv em: {dir_in}");  return

    dfs_agregados = []
    for arquivo in arquivos:
        df_temp = ler_csv_seguro(arquivo)
        if df_temp.empty:
            continue

        col_dt = next((c for c in df_temp.columns
                       if any(k in c.lower() for k in ('instante','data','referencia'))), None)
        if col_dt:
            df_temp = criar_tempo(df_temp, col_dt)

        cols_vals = [c for c in df_temp.columns if c.startswith('val_')]
        if not cols_vals:
            cols_vals = [c for c in df_temp.columns
                         if df_temp[c].dtype in (float,'float32','float64')
                         and 'KEY' not in c]

        for c in cols_vals:
            if df_temp[c].dtype == object:
                df_temp[c] = pd.to_numeric(
                    df_temp[c].astype(str).str.replace(',','.'), errors='coerce')
            df_temp[c] = df_temp[c].astype('float32')

        col_sub = next((c for c in df_temp.columns
                        if 'subsistema' in c.lower() or 'submercado' in c.lower()), None)
        if not col_sub:
            UI.warning(f"Coluna de subsistema ausente em {os.path.basename(arquivo)}");  continue

        grp = [c for c in [col_sub,"KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA"]
               if c in df_temp.columns]
        df_agg = df_temp.groupby(grp, as_index=False)[cols_vals].sum()
        df_agg.rename(columns={col_sub: 'KEY_SUBMERCADO'}, inplace=True)
        df_agg.rename(columns={c: c.upper() for c in cols_vals}, inplace=True)
        dfs_agregados.append(df_agg)
        del df_temp, df_agg;  gc.collect()

    if not dfs_agregados:
        UI.warning("Nenhum dado válido para TERMICA");  return

    df_final      = pd.concat(dfs_agregados, ignore_index=True)
    cols_vals_up  = [c for c in df_final.columns if c.startswith('VAL_')]
    df_final = df_final.groupby(
        ["KEY_SUBMERCADO","KEY_ANO","KEY_MES","KEY_DIA","KEY_HORA"],
        as_index=False)[cols_vals_up].sum()
    df_final['KEY_SUBMERCADO'] = (df_final['KEY_SUBMERCADO'].astype(str)
                                                              .str.upper()
                                                              .replace(SIGLA_TO_SUB))

    salvar_por_submercado(df_final, dir_out, "geracao-termica",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df_final;  gc.collect()


# ── 17 – Meteorologia INMET ───────────────────────────────────────────────────
def processar_inmet():
    """
    Processa os dados meteorológicos das estações do INMET.
    Fluxo por submercado:
      1. Varre as pastas anuais (ex.: /11-INMET-METEOROLOGICOS/2021/)
      2. Identifica o submercado de cada arquivo pelo código UF no nome
      3. Agrega médias horárias por estação (cod_estacao como sufixo)
      4. Consolida todas as estações em um único DataFrame via concat lateral
      5. Salva um CSV por submercado com prefixo METEO_

    Gerenciamento de memória:
      Estações são concatenadas em chunks de 30 para evitar MemoryError
      no Google Colab. gc.collect() é chamado a cada 100 arquivos.
    """
    chave  = "INMET"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando INMET – Pastas por Ano (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]
    os.makedirs(dir_out, exist_ok=True)

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta INMET não encontrada: {dir_in}");  return

    def limpar_str(s):
        """Normaliza string para snake_case sem acentos e em maiúsculas."""
        if not isinstance(s, str):
            s = str(s)
        s = ''.join(c for c in unicodedata.normalize('NFD', s)
                    if unicodedata.category(c) != 'Mn')
        return s.upper().replace(' ','_').replace('-','_').replace('.','').strip()

    # Mapeamento UF → submercado elétrico
    UF_TO_SUB = {
        'AC':'NORTE','AM':'NORTE','AP':'NORTE','PA':'NORTE','RO':'NORTE','RR':'NORTE','TO':'NORTE',
        'AL':'NORDESTE','BA':'NORDESTE','CE':'NORDESTE','MA':'NORDESTE','PB':'NORDESTE',
        'PE':'NORDESTE','PI':'NORDESTE','RN':'NORDESTE','SE':'NORDESTE',
        'DF':'SUDESTE','ES':'SUDESTE','GO':'SUDESTE','MG':'SUDESTE','MS':'SUDESTE',
        'MT':'SUDESTE','RJ':'SUDESTE','SP':'SUDESTE',
        'PR':'SUL','RS':'SUL','SC':'SUL',
    }

    def detectar_submercado_por_arquivo(nome_arquivo):
        """
        Detecta o submercado pelo nome do arquivo INMET.
        Prioriza o código de região após 'INMET' (ex.: INMET_NE_);
        fallback para sigla de UF presente no nome.
        """
        partes = nome_arquivo.upper().replace('.CSV','').split('_')
        for i, p in enumerate(partes):
            if p == 'INMET' and i + 1 < len(partes):
                sigla = partes[i + 1]
                mapa_sigla = {'N':'NORTE','NE':'NORDESTE','S':'SUL','SE':'SUDESTE','CO':'SUDESTE'}
                if sigla in mapa_sigla:
                    return mapa_sigla[sigla]
                if sigla in UF_TO_SUB:
                    return UF_TO_SUB[sigla]
        for p in partes:
            if p in UF_TO_SUB:
                return UF_TO_SUB[p]
        return None

    for submercado_alvo in SUBMERCADOS:
        UI.step(f"   ↳ Carregando INMET para: {submercado_alvo}...")
        storage_stations = {}

        for ano in ANOS_ALVO:
            pasta_ano = os.path.join(dir_in, str(ano))
            if not os.path.exists(pasta_ano):
                continue

            csvs_do_ano = [f for f in os.listdir(pasta_ano) if f.lower().endswith('.csv')]

            for i, nome_arq in enumerate(csvs_do_ano):
                if detectar_submercado_por_arquivo(nome_arq) != submercado_alvo:
                    continue

                caminho_csv = os.path.join(pasta_ano, nome_arq)
                try:
                    # Arquivos INMET possuem 8 linhas de cabeçalho a pular
                    df = pd.read_csv(caminho_csv, sep=';', encoding='latin1',
                                     skiprows=8, on_bad_lines='skip', low_memory=True)

                    c_data = next((c for c in df.columns if 'DATA' in c.upper()), None)
                    c_hora = next((c for c in df.columns
                                   if 'HORA' in c.upper() and 'UTC' in c.upper()), None)
                    if not c_hora:
                        c_hora = next((c for c in df.columns if 'HORA' in c.upper()), None)
                    if not c_data or not c_hora:
                        continue

                    df = df.dropna(subset=[c_data, c_hora])
                    df[c_data] = pd.to_datetime(df[c_data], errors='coerce')
                    hora_str   = (df[c_hora].astype(str)
                                             .str.replace(' UTC','',regex=False)
                                             .str.replace(':','',regex=False)
                                             .str.zfill(4).str[:2])
                    df['KEY_HORA'] = pd.to_numeric(hora_str, errors='coerce').astype('Int16')
                    df['KEY_ANO']  = df[c_data].dt.year.astype('Int16')
                    df['KEY_MES']  = df[c_data].dt.month.astype('Int8')
                    df['KEY_DIA']  = df[c_data].dt.day.astype('Int8')

                    # Extrai código da estação do nome do arquivo (padrão Xnnn)
                    partes_nome = nome_arq.upper().replace('.CSV','').split('_')
                    cod_estacao = next((p for p in partes_nome if re.match(r'^[A-Z]\d{3}$',p)),
                                      partes_nome[-1] if partes_nome else "UNK")

                    col_map   = {}
                    cols_keys = ['KEY_ANO','KEY_MES','KEY_DIA','KEY_HORA']
                    for c in df.columns:
                        if c in [c_data, c_hora] + cols_keys:
                            continue
                        nome_var  = limpar_str(c.split('(')[0])
                        col_dest  = f"METEO_{nome_var}_{cod_estacao}"
                        col_map[c] = col_dest
                        if df[c].dtype == object:
                            df[c] = pd.to_numeric(
                                df[c].astype(str).str.replace(',','.'),
                                errors='coerce').astype('float32')
                        else:
                            df[c] = df[c].astype('float32')

                    df.rename(columns=col_map, inplace=True)
                    df_final = df[cols_keys + list(col_map.values())].copy()
                    df_final = df_final.groupby(cols_keys).mean()

                    if cod_estacao not in storage_stations:
                        storage_stations[cod_estacao] = df_final
                    else:
                        storage_stations[cod_estacao] = pd.concat(
                            [storage_stations[cod_estacao], df_final])
                        storage_stations[cod_estacao] = (
                            storage_stations[cod_estacao]
                            .groupby(level=[0,1,2,3]).mean())
                except Exception as e:
                    UI.warning(f"  Erro ao ler {nome_arq}: {e}")

                if i % 100 == 0:
                    gc.collect()
            gc.collect()

        if not storage_stations:
            UI.register_warning("meteorologia", submercado_alvo);  continue

        UI.step(f"     Consolidando {submercado_alvo} ({len(storage_stations)} estações)...")
        STATS[submercado_alvo]["estacoes"] = len(storage_stations)

        try:
            lista_dfs  = list(storage_stations.values())
            chunk_size = 30
            chunks     = [lista_dfs[i:i+chunk_size] for i in range(0,len(lista_dfs),chunk_size)]
            df_parcial = pd.DataFrame()
            for chunk in chunks:
                df_parcial = (pd.concat(chunk, axis=1) if df_parcial.empty
                              else pd.concat([df_parcial]+chunk, axis=1))
                gc.collect()

            df_parcial.reset_index(inplace=True)
            df_parcial.insert(0, 'KEY_SUBMERCADO', submercado_alvo)

            salvar_por_submercado(
                df_parcial, dir_out, "meteorologia",
                forcar_sub=submercado_alvo,
                chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="CLIMA")
        except MemoryError:
            UI.error(f"Memória insuficiente para consolidar INMET {submercado_alvo}.")
        del storage_stations, lista_dfs, df_parcial;  gc.collect()


# ── 18 – Carga Verificada (API) ───────────────────────────────────────────────
def processar_carga_verificada():
    """
    Processa a Carga Verificada coletada via API REST da ONS.
    Aplica filtro de minuto == 0 para garantir frequência horária exata
    e mapeia os códigos de área (SECO, SE/CO, S, NE, N) para submercados.
    """
    chave  = "CARGA_VERIFICADA"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando Carga Verificada (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "carga_verificada")
    if not arquivos:
        UI.warning(f"Sem arquivos carga_verificada*.csv em: {dir_in}");  return

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return

    df = pd.concat(dfs, ignore_index=True)
    col_dt = next((c for c in df.columns
                   if any(k in c.lower() for k in ('referencia','instante','data'))), None)
    if col_dt:
        df = criar_tempo(df, col_dt)
        col_dt_pd = pd.to_datetime(df[col_dt], errors='coerce')
        df = df[col_dt_pd.dt.minute == 0]

    col_sub = next((c for c in df.columns
                    if any(k in c.lower() for k in ('areacarga','subsistema','submercado'))), None)
    mapa_sub = {'SECO':'SUDESTE','SE/CO':'SUDESTE','SE':'SUDESTE',
                'S':'SUL','NE':'NORDESTE','N':'NORTE'}
    if col_sub:
        df['KEY_SUBMERCADO'] = (df[col_sub].astype(str).str.upper()
                                            .str.strip().replace(mapa_sub))
    else:
        UI.warning("Coluna de submercado não encontrada em carga verificada.");  return

    mapa_feats = {}
    for c in df.columns:
        if 'cargaglobal' in c.lower():
            mapa_feats[c] = c.upper()
        elif 'cargasuperv' in c.lower():
            mapa_feats[c] = c.upper()
    if not mapa_feats:
        for c in df.columns:
            if c.startswith('val_'):
                mapa_feats[c] = c.upper()

    cols_val = list(mapa_feats.keys())
    for c in cols_val:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    grp    = [c for c in ['KEY_SUBMERCADO','KEY_ANO','KEY_MES','KEY_DIA','KEY_HORA']
              if c in df.columns]
    df_agg = df.groupby(grp, as_index=False)[cols_val].sum()
    df_agg.rename(columns=mapa_feats, inplace=True)

    salvar_por_submercado(df_agg, dir_out, "carga-verificada",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df, df_agg;  gc.collect()


# ── 19 – Cotação Dólar (BCB) ──────────────────────────────────────────────────
def processar_dolar():
    """
    Processa a cotação PTAX do dólar americano (Banco Central do Brasil).
    Realiza broadcast diário × 24 horas × 4 submercados, replicando o valor
    diário da cotação para todas as horas e regiões (feature global).
    """
    chave  = "DOLAR"
    num_ds = DATASET_NUM[chave]
    fonte  = DATASET_FONTE[chave]
    UI.step(f"[{num_ds}] Processando Cotação Dólar (fonte: {fonte})...")
    dir_in, dir_out = MAPA_PASTAS[chave]

    if not os.path.exists(dir_in):
        UI.warning(f"Pasta não encontrada: {dir_in}");  return

    arquivos = listar_csvs(dir_in, "dolar")
    if not arquivos:
        UI.warning(f"Sem arquivos dolar*.csv em: {dir_in}");  return

    dfs = [ler_csv_seguro(f) for f in arquivos]
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return

    df = pd.concat(dfs, ignore_index=True)
    col_data = next((c for c in df.columns
                     if any(k in c.lower() for k in ('data','date','cotacao'))), None)
    if col_data:
        df[col_data] = pd.to_datetime(df[col_data], errors='coerce')
        df['KEY_ANO'] = df[col_data].dt.year
        df['KEY_MES'] = df[col_data].dt.month
        df['KEY_DIA'] = df[col_data].dt.day
    else:
        UI.warning(f"Coluna de data não encontrada no Dólar. Colunas: {list(df.columns)}");  return

    col_compra = next((c for c in df.columns if 'compra' in c.lower()), None)
    col_venda  = next((c for c in df.columns if 'venda'  in c.lower()), None)

    feat_map = {}
    for x, alias in [(col_compra, 'DOLAR_VALOR_COMPRA'), (col_venda, 'DOLAR_VALOR_VENDA')]:
        if x and x in df.columns:
            df[x] = pd.to_numeric(df[x].astype(str).str.replace(',','.'), errors='coerce')
            feat_map[x] = alias

    if not feat_map:
        UI.warning("Nenhuma coluna de compra/venda encontrada.");  return

    feat_cols = list(feat_map.keys())
    df_b = df[['KEY_ANO','KEY_MES','KEY_DIA'] + feat_cols].drop_duplicates()
    df_b.rename(columns=feat_map, inplace=True)

    # Broadcast: cross-join com 24 horas e 4 submercados
    h = pd.DataFrame({'KEY_HORA': range(24)})
    s = pd.DataFrame({'KEY_SUBMERCADO': SUBMERCADOS})
    df_b['_k'] = 1;  h['_k'] = 1;  s['_k'] = 1
    df_x = pd.merge(df_b, h, on='_k').drop('_k', axis=1)
    df_x['_k'] = 1
    df_final = pd.merge(df_x, s, on='_k').drop('_k', axis=1)

    salvar_por_submercado(df_final, dir_out, "dolar",
                          chave_ds=chave, num_ds=num_ds, fonte=fonte, tipo_dado="FEATURE")
    del df, df_b, df_x, df_final;  gc.collect()


# ==============================================================================
# 🚀 MAIN  –  Execução na ordem padronizada (01 → 19)
# ==============================================================================
def main():
    UI.banner()
    start_total = time.time()

    # 01 – PLD (CCEE) — variável-alvo (target) do modelo preditivo
    processar_pld()

    # 02 – Intercâmbio Nacional (ONS) — saldo de fluxo entre subsistemas
    processar_intercambio()

    # 03 – Balanço Energético (ONS) — geração por fonte + carga horária
    etl_simples(
        "BALANCO", "balanco_",
        {
            'val_gerhidraulica': 'VAL_GERHIDRAULICA',
            'val_gertermica':    'VAL_GERTERMICA',
            'val_gereolica':     'VAL_GEREOLICA',
            'val_gersolar':      'VAL_GERSOLAR',
            'val_carga':         'VAL_CARGA',
        },
        "balanco-energia",
        chave_ds="BALANCO", tipo_dado="FEATURE"
    )

    # 04 – CMO (ONS) — custo marginal semi-horário resampleado para hora cheia
    etl_simples(
        "CMO", "cmo_",
        {'val_cmo': 'VAL_CMO'},
        "cmo",
        filtro_hora00=True,
        chave_ds="CMO", freq="Semi-Horária", tipo_dado="FEATURE"
    )

    # 05 – Curva de Carga (ONS) — carga de energia horária por subsistema
    etl_simples(
        "CURVA", "curva_",
        {'val_cargaenergiahomwmed': 'VAL_CARGAENERGIAMWMED'},
        "curva-carga",
        chave_ds="CURVA", tipo_dado="FEATURE"
    )

    # 06 – EAR (ONS) — energia armazenada diária expandida para horária
    expandir_diario("EAR", "ear_",
                    ["ear_verif_subsistema_percentual","val_ear","ear"])

    # 07 – ENA (ONS) — energia natural afluente diária expandida para horária
    expandir_diario("ENA", "ena_",
                    ["ena_bruta_regiao_mwmed","val_ena","ena"])

    # 08 – Carga (ONS) — carga de energia diária expandida para horária
    expandir_diario("CARGA", "carga_",
                    ["val_cargaenergiamwmed","val_carga","carga"])

    # 09 – CVU (ONS) — custo variável semanal explodido em horas (média)
    processar_cvu()

    # 10 – Volume de Espera (ONS) — volume diário de reservatórios expandido para horário
    expandir_diario("VOLUME_ESPERA", "volume_espera_",
                    ["val_volumeesperapercentual","val_volume","volume"])

    # 11 – Dados Hidrológicos (ONS) — vazões e volumes horários por reservatório
    etl_simples(
        "HIDRO", "hidro_",
        {
            'val_volumeutil':     'VAL_VOLUMEUTIL',
            'val_vazaoafluente':  'VAL_VAZAOAFLUENTE',
            'val_vazaoturbinada': 'VAL_VAZAOTURBINADA',
            'val_vazaovertida':   'VAL_VAZAOVERTIDA',
        },
        "hidro-reservatorios",
        chave_ds="HIDRO", tipo_dado="FEATURE"
    )

    # 12 – Disponibilidade de Usinas (ONS) — potência instalada mensal → horária
    etl_simples(
        "DISPONIBILIDADE", "disponibilidade_",
        {'val_potenciainstalada': 'VAL_POTENCIAINSTALADA'},
        "disponibilidade",
        chave_ds="DISPONIBILIDADE", freq="Mensal", tipo_dado="FEATURE"
    )

    # 13 – Fator de Capacidade (ONS) — fatores horários de capacidade por usina
    processar_fator_capacidade()

    # 14 – Geração de Usinas (ONS) — geração total horária por usina e subsistema
    etl_simples(
        "GERACAO", "geracao_usina_",
        {'val_geracao': 'VAL_GERACAO'},
        "geracao-usina",
        chave_ds="GERACAO", tipo_dado="FEATURE"
    )

    # 15 – Energia Vertida (ONS) — energia vertida e turbinável horária
    etl_simples(
        "VERTIDA", "energia_vertida_",
        {'val_energiavertida': 'VAL_ENERGIAVERTIDA'},
        "energia-vertida",
        chave_ds="VERTIDA", tipo_dado="FEATURE"
    )

    # 16 – Geração Térmica de Despacho (ONS) — geração horária com auto-detect de colunas
    processar_termica()

    # 17 – Meteorologia INMET — agregação espacial de estações em médias regionais horárias
    processar_inmet()

    # 18 – Carga Verificada via API (ONS) — carga real verificada horária por subsistema
    processar_carga_verificada()

    # 19 – Cotação Dólar PTAX (BCB) — broadcast diário × 24h × 4 submercados
    processar_dolar()

    # ── Impressão dos relatórios de auditoria ─────────────────────────────────
    UI.print_tabela_2()
    UI.print_tabela_3()
    UI.print_tabela_4()

    if UI.WARNINGS_BUFFER:
        print(f"\n{UI.YELLOW}⚠ Avisos registrados:{UI.RESET}")
        for w in UI.WARNINGS_BUFFER:
            print(f"   {UI.YELLOW}• {w}{UI.RESET}")

    elapsed = time.time() - start_total
    print(f"\n{UI.CYAN}➤ Tempo Total: {elapsed:.2f}s{UI.RESET}")
    print(f"{UI.GREEN}✅ PIPELINE CONCLUÍDO COM SUCESSO!{UI.RESET}")


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    main()